# Мне не нравится стиль Ивлева с комментариями в Python ячейках, поэтому они в Markdown. Не попадитесь на этом)

# Критерий хи-квадрта

In [230]:
import pandas as pd
import numpy as np
import scipy.stats as ss

df = pd.read_csv("../datasets/heart.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,target
0,63,1,3,145,233,1,0,150,0,1
1,37,1,2,130,250,0,1,187,0,1
2,41,0,1,130,204,0,0,172,0,1
3,56,1,1,120,236,0,1,178,0,1
4,57,0,0,120,354,0,1,163,1,1


Сформулируем гипотезу: будем смотреть на переменные Sex (1 - мужчина, 0 - женщина). Target - есть ли сердечно-сосудистое заболевание (1 - да, 0 - нет). 

H0 - распределение по категориям независимо, наличие заболеваний не зависит от пола

In [231]:
ct_o = df.groupby("sex")["target"].value_counts().unstack()  # таблица сопряженности
ct_o

target,0,1
sex,,
0,24,72
1,114,93


Напишите, что означают значения в таблице выше?

Это сгруппированные значения по полу и наличию болезней. Если в <b>"Статистике и Котиках"</b> (25 страница, зависит от издания) - таблица эмпирических значений.

### Расчитайте количество степеней свободы по формуле ниже и запишите в переменную dof.

$
dof = (R - 1)(C - 1)
$

In [232]:
dof = (ct_o.shape[0] - 1) * (ct_o.shape[1] - 1)  # по формуле выше
dof

1

In [233]:
alpha = 0.01  # уровень значимости, вероятность получить ошибку первого рода

# найдите таблицу для расчёта критических значений критерия хи и запишите его в переменную ниже
# сделаем вид, что я полез в таблицу, а так возьмём из ss
critical_value = ss.chi2.ppf(1 - alpha, dof)
critical_value

np.float64(6.6348966010212145)

### Расчитайте теоретические частоты. Это можно сделать по примеру из презентации.
### Сравните полученый результат с результатом из формулы ниже. Они должны сойтись.

$
f_e = \frac{f_cf_r}{n}
$

In [234]:
# тут просто - сумма по строкам умножить на сумму по столбцам, делить на сумму по всей таблице
ct = pd.DataFrame(
    np.outer(ct_o.sum(axis=1), ct_o.sum(axis=0)) / ct_o.values.sum(),
    index=ct_o.index,
    columns=ct_o.columns,
)
ct

target,0,1
sex,,
0,43.722772,52.277228
1,94.277228,112.722772


In [ ]:
# тут 44, если вы запустили весь код и потеряли это значение
round(ct[0].sum() * ct.iloc[0].sum() / n_total)

44

$
\chi^2 = \sum{\frac{(f_o - f_e)^2}{f_e}}
$

In [235]:
# Расчитайте Xi^2

chi_square = ((ct_o - ct).pow(2) / ct).values.sum()
print(chi_square)  # bruh так делать в Jupyter, тут без print надо

23.914383914761988


### Сделайте финальный вывод, можем ли мы отвергнуть нулевую гипотезу?

In [236]:
chi_square > critical_value

np.True_

In [ ]:
# для самопроверки

# вообще тут хи-квадрат и pvalue считаются, но числа отличаются от нижних и наших
# скорее всего проблема в степенях и float
_, _, dof_sp, ct_sp = ss.chi2_contingency(ct_o)

chi_square_sp, pvalue = ss.chisquare(ct_o, ct, ddof=dof, axis=None)

np.allclose(chi_square, chi_square_sp), np.allclose(dof, dof_sp), np.allclose(ct, ct_sp)

(True, True, True)

In [238]:
pvalue < alpha  # алтернатива критическому значению

np.True_

## По приколу то же самое, но на чистом питоне, чтобы лучше понимать как и что

In [198]:
ct_o = ct_o.to_numpy().tolist()
ct_o

[[24, 72], [114, 93]]

In [199]:
critical_value = critical_value  # загугли таблицу для Хи-квадрат
ct = [[0 for _ in range(len(ct_o))] for _ in range(len(ct_o[0]))]
ct_oT = list(zip(*ct_o))

for i in range(len(ct_o)):
    for j in range(len(ct_o[0])):
        ct[j][i] = (
            sum(ct_o[j : j + 1][0])
            * sum(ct_oT[i : i + 1][0])
            / sum(sum(row) for row in ct_o)
        )

ct

[[43.722772277227726, 52.277227722772274],
 [94.27722772277228, 112.72277227722772]]

In [200]:
# понт
[
    [
        sum(ct_o[j : j + 1][0])
        * sum(ct_oT[i : i + 1][0])
        / sum(sum(row) for row in ct_o)
        for i in range(len(ct_o))
    ]
    for j in range(len(ct_o[0]))
]

[[43.722772277227726, 52.277227722772274],
 [94.27722772277228, 112.72277227722772]]

In [201]:
chi = [[0 for _ in range(len(ct_o))] for _ in range(len(ct_o[0]))]

for i in range(len(ct_o)):
    for j in range(len(ct_o[0])):
        chi[j][i] = ((ct_o[i][j] - ct[i][j]) ** 2) / (ct[i][j])

chi_square = sum(sum(row) for row in chi)
chi_square, np.allclose(chi_square, chi_square_sp)

(23.914383914761988, True)

# T-test

In [202]:
a = [8, 5, 7, 8, 8, 10, 9, 6, 2]
b = [4, 8, 2, 4, 8, 7, 9, 8, 4, 4, 8]

Мы будем делать без $\mu$, потому что нулевая гипотеза - средние равны, $\mu_1 = \mu_2 => \mu_1 - \mu_2 = 0$

$$t = \frac{(M_1 - M_2) - (\mu_1 - \mu_2)}{S_{M_1 - M_2}}$$

## NumPy way

In [223]:
a_np = np.array(a)
b_np = np.array(b)

# объединённое стандартное отклонение
S = np.sqrt(a_np.var(ddof=1) / len(a_np) + b_np.var(ddof=1) / len(b_np))

t_np = (a_np.mean() - b_np.mean()) / S
t_np

np.float64(0.926020558819996)

## Pure Python

In [224]:
a_mean = sum(a) / len(a)
b_mean = sum(b) / len(b)

a_var = sum((a[i] - a_mean) ** 2 for i in range(len(a))) / (len(a) - 1)
b_var = sum((b[i] - b_mean) ** 2 for i in range(len(b))) / (len(b) - 1)

S = (a_var / len(a) + b_var / len(b)) ** 0.5
t_py = (a_mean - b_mean) / S
t_py

0.926020558819996

## SciPy

In [226]:
t_sp, pvalue = ss.ttest_ind(a, b)
t_np, t_py, t_sp

(np.float64(0.926020558819996),
 0.926020558819996,
 np.float64(0.9255975201083726))

## Степени свободы

In [206]:
alpha = 0.05
dof = len(a) - 1 + len(b) - 1

critical_value = ss.t.ppf(1 - alpha / 2, dof)
critical_value

np.float64(2.10092204024096)

In [207]:
# ну и
t_np > critical_value, pvalue < alpha

(np.False_, np.False_)